# Phase 2 — Load & Inspect Real Fit Dataset

Source: Kaggle "Clothing Fit Dataset for Size Recommendation" (ModCloth + RentTheRunway).

This notebook only **inspects** the raw data (shape, dtypes, nulls, class balance) —
no cleaning, imputation, or feature engineering happens here. That's Phase 3.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

MODCLOTH_PATH = "../data/raw/clothing_fit/modcloth_final_data.json"
RENTTHERUNWAY_PATH = "../data/raw/clothing_fit/renttherunway_final_data.json"

modcloth = pd.read_json(MODCLOTH_PATH, lines=True)
renttherunway = pd.read_json(RENTTHERUNWAY_PATH, lines=True)

modcloth.shape, renttherunway.shape

((82790, 18), (192544, 15))

## ModCloth

In [2]:
modcloth.dtypes

item_id             int64
waist             float64
size                int64
quality           float64
cup size              str
hips              float64
bra size          float64
category              str
bust                  str
height                str
user_name             str
length                str
fit                   str
user_id             int64
shoe size         float64
shoe width            str
review_summary        str
review_text           str
dtype: object

In [3]:
nulls = modcloth.isnull().sum().sort_values(ascending=False)
pct = (modcloth.isnull().mean() * 100).round(1).sort_values(ascending=False)
pd.DataFrame({"n_null": nulls, "pct_null": pct})

,n_null,pct_null
bra size,6018,7.3
bust,70936,85.7
category,0,0.0
cup size,6255,7.6
fit,0,0.0
height,1107,1.3
hips,26726,32.3
item_id,0,0.0
length,35,0.0
quality,68,0.1


In [4]:
modcloth.head(3)

,item_id,waist,size,quality,cup size,hips,bra size,category,bust,height,user_name,length,fit,user_id,shoe size,shoe width,review_summary,review_text
0,123373,29.0,7,5.0,d,38.0,34.0,new,36,5ft 6in,Emily,just right,small,991571,NaN,NaN,NaN,NaN
1,123373,31.0,13,3.0,b,30.0,36.0,new,NaN,5ft 2in,sydneybraden2001,just right,small,587883,NaN,NaN,NaN,NaN
2,123373,30.0,7,2.0,b,NaN,32.0,new,NaN,5ft 7in,Ugggh,slightly long,small,395665,9.0,NaN,NaN,NaN


In [5]:
modcloth["fit"].value_counts(dropna=False)

fit
fit      56757
large    13059
small    12974
Name: count, dtype: int64

In [6]:
modcloth["category"].value_counts(dropna=False)

category
new          21488
tops         20364
dresses      18650
bottoms      15266
outerwear     4223
sale          2524
wedding        275
Name: count, dtype: int64

## RentTheRunway

In [7]:
renttherunway.dtypes

fit                   str
user_id             int64
bust size             str
item_id             int64
weight                str
rating            float64
rented for            str
review_text           str
body type             str
review_summary        str
category              str
height                str
size                int64
age               float64
review_date           str
dtype: object

In [8]:
nulls = renttherunway.isnull().sum().sort_values(ascending=False)
pct = (renttherunway.isnull().mean() * 100).round(1).sort_values(ascending=False)
pd.DataFrame({"n_null": nulls, "pct_null": pct})

,n_null,pct_null
age,960,0.5
body type,14637,7.6
bust size,18411,9.6
category,0,0.0
fit,0,0.0
height,677,0.4
item_id,0,0.0
rating,82,0.0
rented for,10,0.0
review_date,0,0.0


In [9]:
renttherunway.head(3)

,fit,user_id,bust size,item_id,weight,rating,rented for,review_text,body type,review_summary,category,height,size,age,review_date
0,fit,420272,34d,2260466,137lbs,10.0,vacation,An adorable romper! Belt and zipper were a lit...,hourglass,So many compliments!,romper,"5' 8""",14,28.0,"April 20, 2016"
1,fit,273551,34b,153475,132lbs,10.0,other,I rented this dress for a photo shoot. The the...,straight & narrow,I felt so glamourous!!!,gown,"5' 6""",12,36.0,"June 18, 2013"
2,fit,360448,NaN,1063761,NaN,10.0,party,This hugged in all the right places! It was a ...,NaN,It was a great time to celebrate the (almost) ...,sheath,"5' 4""",4,116.0,"December 14, 2015"


In [10]:
renttherunway["fit"].value_counts(dropna=False)

fit
fit      142058
small     25779
large     24707
Name: count, dtype: int64

In [11]:
renttherunway["body type"].value_counts(dropna=False)

body type
hourglass            55349
athletic             43667
pear                 22135
petite               22131
full bust            15006
straight & narrow    14742
NaN                  14637
apple                 4877
Name: count, dtype: int64

## Observations (no action taken yet, Phase 3 handles these)

- **ModCloth** (82,790 rows, 18 cols): `waist` (96.5%), `bust` (85.7%), `shoe width`
  (77.5%), `shoe size` (66.3%) are mostly null — shoe fields are irrelevant to garment
  sizing and will likely be dropped. `hips` is 32.3% null, `cup size`/`bra size` ~7%
  null, `height` only 1.3% null. No `weight` or `body type` column exists in ModCloth
  at all (confirms the Master Plan's noted gap).
- **RentTheRunway** (192,544 rows, 15 cols): `weight` 15.6% null, `bust size` 9.6%
  null, `body type` 7.6% null — otherwise fairly complete.
- **Target class balance** (`fit`): both datasets are skewed toward "fit" (ModCloth
  68.5%, RentTheRunway 73.8%), consistent with the Master Plan's expectation —
  reinforces the choice of F1-macro over raw accuracy for model selection.
- **Schema mismatch across the two sources**: column names/formats differ (e.g.
  `bust` free-text in ModCloth vs `bust size` band+cup string in RentTheRunway,
  `height` as `"5ft 6in"` vs `"5' 8\""`) — will need harmonizing in Phase 3
  before merging into one `model_ready` dataset.